In [1]:
#context aware splitting-we are keeping the context and doing the splitting operation

In [2]:
import os
from git import Repo

# Document loaders & parsers
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser

# Splitters & vectorstore
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# Gemini LLM (still used for chat) + HuggingFace Embeddings (local, no quota)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings

# Chains & Memory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationSummaryMemory

C:\Users\victu\AppData\Local\Temp\ipykernel_9760\2614523312.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.generic import GenericLoader
c:\Users\victu\miniconda3\envs\llmapp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
!mkdir -p test_repo


A subdirectory or file -p already exists.
Error occurred while processing: -p.
A subdirectory or file test_repo already exists.
Error occurred while processing: test_repo.


In [5]:
repo_path = 'test_repo'
repo = Repo.clone_from("https://github.com/msiemens/tinydb", to_path=repo_path )

Loading data from files

In [6]:
%pwd

'c:\\Users\\victu\\OneDrive\\Documents\\Generative AI campusX\\projects\\RealTime-Source-Code-Analyzer Project\\research'

In [7]:
loader = GenericLoader.from_filesystem(repo_path,
                                        glob = "**/*",
                                       suffixes=[".py"],
                                       parser = LanguageParser(language=Language.PYTHON, parser_threshold=500)
)

In [8]:
documents=loader.load()

In [9]:
documents
len(documents)

94

In [10]:
# Splitting documents into chunks
documents_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, 
    chunk_size=1500, 
    chunk_overlap=150
)

In [11]:
text=documents_splitter.split_documents(documents)

In [12]:
len(text)

179

In [13]:
# Using HuggingFace local embeddings -- no API key, no quota limits
from langchain_huggingface import HuggingFaceEmbeddings

# BAAI/bge-small-en-v1.5: fast, high-quality, great for code retrieval
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cpu'},        # change to 'cuda' if you have a GPU
    encode_kwargs={'normalize_embeddings': True}  # recommended for bge models
)
print('Embedding model loaded successfully!')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2865.96it/s]


Embedding model loaded successfully!


In [14]:
# Pass documents and embedding function into Chroma
vectordb = Chroma.from_documents(text, embedding=embedding, persist_directory='./data')

In [15]:
llm=ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
)

In [16]:
memory=ConversationSummaryMemory(
    llm=llm,
    memory_key='chat_history',
    return_messages=True
)

C:\Users\victu\AppData\Local\Temp\ipykernel_9760\1691646943.py:1: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory=ConversationSummaryMemory(


In [17]:
retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k":8})

In [18]:
qa = ConversationalRetrievalChain.from_llm(llm,retriever=retriever , memory=memory)

In [19]:
question="What is TinyDB and what is it used for?"

In [20]:
qa(question)

C:\Users\victu\AppData\Local\Temp\ipykernel_9760\4101529553.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  qa(question)
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'question': 'What is TinyDB and what is it used for?',
 'chat_history': [SystemMessage(content='', additional_kwargs={}, response_metadata={})],
 'answer': 'Based on the provided context, **TinyDB** is a lightweight, document-oriented database for Python. \n\nIt is used for:\n* **Storing Python Data:** It stores various Python data types using configurable storage mechanisms (such as `JSONStorage` for writing to disk or `MemoryStorage` for in-memory databases).\n* **Organizing Data:** It allows you to organize and store data across multiple tables.\n* **Querying Data:** It provides a rich syntax (both ORM-like queries and classic `where()` syntax) to search and filter stored documents using binary operators like `AND` (`&`) and `OR` (`|`).\n* **Data Operations:** It supports built-in operations for modifying stored documents, such as setting, deleting fields, incrementing/decrementing values, or adding/subtracting numbers and strings.'}

In [ ]:
print(result['answer'])